<a href="https://colab.research.google.com/github/Innovatewithapple/YoutubeSummarizationNLP/blob/main/YoutubeSummaryNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 25.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.1 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=d6ba6b14df968815a013cb0d924bd9e210ab98e8b750310b688c8044e0be002a
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [3]:
import yt_dlp
import whisper
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer,AutoModelForCausalLM
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from torch.utils.data import DataLoader,Dataset

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
#-----------Transcribe Model-----------!
transcribe_Model = whisper.load_model('small')

#--------Decoder-------!
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

In [ ]:
def download_audio(youtube_url, output_path="audio"):
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{output_path}/%(title)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        filename = ydl.prepare_filename(info)
        # Return the actual .mp3 path
        return filename.rsplit('.', 1)[0] + '.mp3'

In [ ]:
audio_path = download_audio("https://www.youtube.com/watch?v=RRKwmeyIc24")
print(f"Audio saved to: {audio_path}")

In [ ]:
#--------Transcribe Audio----------!
transcribe_result = transcribe_Model.transcribe(audio_path)
transcript = transcribe_result['text']
print(f'Transcript: \n{transcript}')

In [ ]:
def Chunk_process(text,token_size,overlap):
  sentences = sent_tokenize(text)
  sent_token_counts = [len(decoder_tokenizer.encode(sent,add_special_tokens=False)) for sent in sentences]

  chunks = []
  current_chunk = []
  current_len = 0

  for sent,token_len in zip(sentences,sent_token_counts):
    if current_len + token_len <= token_size:
      current_chunk.append(sent) # because sent itself a senetence not character so no need to use " "
      current_len += token_len
    else:
      if current_chunk:
        chunks.append(" ".join(current_chunk))
      current_chunk = current_chunk[-overlap:] + [sent]
      current_len = token_len

  if current_chunk:
    chunks.append(" ".join(current_chunk))

  return chunks

In [ ]:
all_chunks = Chunk_process(transcript,token_size=400,overlap=1)
print(all_chunks[0])

In [4]:
def summarizing_chunk(chunk):
  message = [
      {"role":"user","content":f"summarize this transcript excerpt concisely:\n\n{chunk}"}
  ]
  prompt = decoder_tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)

  inputs = decoder_tokenizer(prompt,return_tensors='pt').to(llm.device)

  with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens = 200,
        do_sample=True,
        temprature=0.7,
        top_p=0.9
    )

    decoded = decoder_tokenizer.decode(outputs[0],skip_special_tokens=True)
    return decoded[len(prompt):]